In [ ]:
import requests

def pull_info(url):
  """
  For single tests of api call
  """
  params = {
    "format": "json",
    "api_key": api_key,
  }
  response = requests.get(url, params=params)
  return response.json()

In [ ]:
hr_119 = pull_info('https://api.congress.gov/v3/bill/119/hr?format=json')

In [ ]:
hr_119

In [ ]:
import requests
import time

API_KEY = api_key  # replace with your key

BASE_URL = "https://api.congress.gov/v3/bill"

def build_text_url(congress, bill_type, number):
    bill_type = bill_type.lower()  # API expects lowercase (hr, s, etc.)
    return f"{BASE_URL}/{congress}/{bill_type}/{number}/text?format=json&api_key={API_KEY}"


def extract_full_bill_text(congress, bill_type, number):
    url = build_text_url(congress, bill_type, number)
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Failed for {bill_type} {number}")
        return None

    data = response.json()

    try:
        # Navigate textVersions structure
        text_versions = data.get("textVersions", [])
        if not text_versions:
            return None

        # Get most recent version
        latest_version = text_versions[0]

        # Usually formatted text link is inside formats
        formats = latest_version.get("formats", [])
        for fmt in formats:
            if fmt.get("type") == "Formatted Text":
                text_url = fmt.get("url")

                # Now fetch actual text
                text_response = requests.get(text_url)
                return text_response.text

        return None

    except Exception as e:
        print("Error parsing text:", e)
        return None


def process_bills(bills_json):
    results = []

    for bill in bills_json.get("bills", []):
        congress = bill.get("congress")
        bill_type = bill.get("type")
        number = bill.get("number")

        full_text = extract_full_bill_text(congress, bill_type, number)

        results.append({
            "bill_number": f"{bill_type} {number}",
            "title": bill.get("title"),
            "full_text": full_text
        })

        time.sleep(0.5)  # prevent rate limiting

    return results

In [ ]:
API_KEY = api_key

def build_text_url(congress, bill_type, number):
    return (
        f"https://api.congress.gov/v3/bill/"
        f"{congress}/{bill_type.lower()}/{number}"
        f"/text?format=json&api_key={API_KEY}"
    )


def test_url_construction(hr_119):
    for bill in hr_119.get("bills", []):
        congress = bill["congress"]
        bill_type = bill["type"]
        number = bill["number"]

        constructed_url = build_text_url(congress, bill_type, number)

        expected_url = (
            f"https://api.congress.gov/v3/bill/"
            f"{congress}/{bill_type.lower()}/{number}"
            f"/text?format=json&api_key={API_KEY}"
        )

        print("Constructed:", constructed_url)
        print("Expected:   ", expected_url)
        print("---")

        assert constructed_url == expected_url

    print("All URLs constructed correctly.")

In [ ]:
import requests
import json
import time

API_KEY = api_key
BASE_URL = "https://api.congress.gov/v3/bill"


def build_text_url(congress, bill_type, number):
    return (
        f"{BASE_URL}/{congress}/{bill_type.lower()}/{number}"
        f"/text?format=json&api_key={API_KEY}"
    )


def fetch_bill_text(congress, bill_type, number):
    url = build_text_url(congress, bill_type, number)
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Failed request for {bill_type} {number}")
        return None

    data = response.json()

    text_versions = data.get("textVersions", [])
    if not text_versions:
        print(f"No text available for {bill_type} {number}")
        return None

    latest_version = text_versions[0]
    formats = latest_version.get("formats", [])

    for fmt in formats:
        if fmt.get("type") == "Formatted Text":
            text_url = fmt.get("url")
            text_response = requests.get(text_url)
            return text_response.text

    return None


def process_and_save(hr_119, output_file="bill_texts.json"):
    results = []

    for bill in hr_119.get("bills", []):
        congress = bill["congress"]
        bill_type = bill["type"]
        number = bill["number"]

        print(f"Fetching {bill_type} {number}...")

        full_text = fetch_bill_text(congress, bill_type, number)

        results.append({
            "congress": congress,
            "bill_number": f"{bill_type} {number}",
            "title": bill.get("title"),
            "full_text": full_text
        })

        time.sleep(0.5)  # avoid rate limiting

    with open(output_file, "w") as f:
        json.dump(results, f, indent=2)

    print(f"Saved to {output_file}")

In [ ]:
process_and_save(hr_119)